# Track B, step 7: LoRA on the larger Granite model (149M), a quick check

**The question:** does a bigger IBM Granite model, fine-tuned with **LoRA**, give a better text score than what we have? This is a one-run check before committing about 4 more hours to the out-of-fold runs the final model would need.

**LoRA in one paragraph:** instead of updating all of Granite's 150 million weights, LoRA freezes them and adds small trainable "adapter" matrices beside the attention and feed-forward layers. Only those adapters and the new classification head learn: **4.0M parameters (2.6%)**. The frozen Granite keeps what it knows about language, and the adapters learn which details matter for exploitation. With so few positives, training fewer weights is also a form of regularization.

**Settings, fixed before running** (`ml/config.py`, `LORA`):

| Setting | Value | Why |
|---|---|---|
| Model | `ibm-granite/granite-embedding-english-r2`, 150M | 3x the small model, same family |
| LoRA rank / alpha / dropout | 16 / 32 / 0.1 | Standard sizes |
| Adapted layers | attention (`Wqkv`, `Wo`) and feed-forward (`Wi`) | All the transformer's large layers |
| Learning rate | 2e-4 | Typical for LoRA, which trains far fewer weights |
| Epochs | 1 | The small model's held-out loss was lowest after epoch 1 (a choice made from training data) |
| Training rows | All 684 positives + 20 negatives each, from 2018-2024 | Same as the small model |

**Compared against, on the 2025 validation set (top 243 of 52 exploited):**

| Text score | hits@243 | AUC |
|---|---|---|
| Frozen small (in the submitted model) | 4 | 0.800 |
| Fully fine-tuned small, 3 epochs | 5 | 0.854 |

**Decision rule, fixed now:** LoRA-149M is worth the ~4 more hours only if it beats the fully fine-tuned small model on **AUC** *and* doesn't lose at the top. That means an AUC above 0.854, and a paired difference in hits@243 whose 90% range reaches at least 0.

The code lives in `ml/text_model.py`; this notebook calls it.

In [1]:
import sys, time, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = next(p for p in [Path(".."), Path(".")] if (p / "ml" / "config.py").exists()).resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
from ml import config, text_model as T

texts = T.load_texts()
train = T.labeled(texts, "train", "2024-12-31")
val = T.labeled(texts, "val", "2025-12-31")
print(f"device: {T.device()} | train 2018-2024: {len(train):,} rows, {train.y.sum()} positives | val 2025: {len(val):,}, {val.y.sum()} positives")

device: mps | train 2018-2024: 165,885 rows, 684 positives | val 2025: 35,281, 52 positives


## Train (one run, about 25-35 minutes on an M3)

In [2]:
t0 = time.time()
model, log = T.fit(train, config.LORA, label="LoRA 149M")
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} of {sum(p.numel() for p in model.parameters()):,}")
print(f"training time: {(time.time() - t0) / 60:.1f} min")
log

{'run': 'LoRA 149M', 'epoch': 1, 'train loss': 0.3688, 'held-out loss': 0.3436, 'held-out AUC': 0.872, 'minutes': 26.4, 'training rows': 12928, 'positives': 604}
trainable parameters: 3,971,330 of 153,577,732
training time: 26.4 min


,run,epoch,train loss,held-out loss,held-out AUC,minutes,training rows,positives
0,LoRA 149M,1,0.3688,0.3436,0.872,26.4,12928,604


## Score 2025 and compare

In [3]:
t0 = time.time()
s_lora = T.predict(model, val)
print(f"scored {len(val):,} rows in {(time.time() - t0) / 60:.1f} min")
ref = lambda f: pd.read_csv(config.TEXT_DIR / f).set_index("cve_id")["text_score"].reindex(val.cve_id).to_numpy()
s_small_ft, s_frozen = ref("text_val_2025_finetuned.csv"), ref("text_val_2025_frozen.csv")
bt = pd.read_parquet(config.DATA_DIR / "backtest/val_2025.parquet").set_index("cve_id").reindex(val.cve_id)
base = (bt.any_exploit * 10 + bt.epss.fillna(0)).to_numpy()
y = val.y.to_numpy(); K = lambda n: int(round(n * config.TOP_FRACTION))
def hits(s, k=None):
    k = k or K(len(y)); tie = np.random.default_rng(0).permutation(len(y)); return int(y[np.lexsort([tie, -np.asarray(s, float)])[:k]].sum())
rows = [{"text score": n, **{f"top {k}": hits(s, k) for k in [243, 500, 1000, 2000, 5000]},
         "AUC": round(float(roc_auc_score(y, s)), 3) if n != "Baseline: exploit code then EPSS" else ""}
        for n, s in [("Baseline: exploit code then EPSS", base), ("Frozen small (submitted)", s_frozen),
                     ("Fine-tuned small, 3 epochs", s_small_ft), ("LoRA 149M, 1 epoch", s_lora)]]
pd.DataFrame(rows).set_index("text score")

scored 35,281 rows in 75.1 min


,top 243,top 500,top 1000,top 2000,top 5000,AUC
text score,,,,,,
Baseline: exploit code then EPSS,14,18,22,24,33,
Frozen small (submitted),4,6,9,19,26,0.8
"Fine-tuned small, 3 epochs",5,8,11,23,35,0.854
"LoRA 149M, 1 epoch",3,5,9,16,38,0.859


In [4]:
def paired(a, b, n=2000, seed=0):
    k = K(len(y)); ra, rb = rankdata(-a, method="ordinal"), rankdata(-b, method="ordinal")
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y)); d.append(y[i][np.argpartition(ra[i], k)[:k]].sum() - y[i][np.argpartition(rb[i], k)[:k]].sum())
    d = np.array(d); return hits(a) - hits(b), int(np.quantile(d, .05)), int(np.quantile(d, .95))
for name, other in [("fine-tuned small", s_small_ft), ("frozen small", s_frozen)]:
    d, lo, hi = paired(s_lora, other)
    print(f"LoRA 149M vs. {name}: {d:+d} hits@243 (90% range {lo:+d} to {hi:+d})")
old = (pd.to_datetime(val.published).dt.year - val.cve_id.str[4:8].astype(int)) >= 2
top = np.lexsort([np.random.default_rng(0).permutation(len(y)), -s_lora])[:K(len(y))]
print(f"LoRA top 243: {int(old.values[top].sum())} old-ID rows, {int((val.assigner.values[top] == 'disclosure@vulncheck.com').sum())} VulnCheck rows "
      f"(fine-tuned small: 45 and 95)")

LoRA 149M vs. fine-tuned small: -2 hits@243 (90% range -6 to +2)


LoRA 149M vs. frozen small: -1 hits@243 (90% range -4 to +1)
LoRA top 243: 21 old-ID rows, 23 VulnCheck rows (fine-tuned small: 45 and 95)


## Decision, by the rule fixed above

In [5]:
auc_lora, auc_ref = roc_auc_score(y, s_lora), roc_auc_score(y, s_small_ft)
d, lo, hi = paired(s_lora, s_small_ft)
go = auc_lora > auc_ref and hi >= 0
print(f"AUC {auc_lora:.3f} vs. {auc_ref:.3f}; hits@243 {d:+d} ({lo:+d} to {hi:+d}) ->",
      "WORTH the out-of-fold runs (about 4 h)" if go else "NOT worth the extra runs: keep the submitted text score")
out = config.TEXT_SCORES["lora-149m"]; out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"cve_id": val.cve_id, "text_score": s_lora}).to_csv(out / "text_val_2025.csv", index=False)
model.save_pretrained(config.TEXT_DIR / "lora149_adapter_2018_2024")
log.to_csv(out / "training_log_val_model.csv", index=False)
print("saved:", sorted(p.name for p in out.iterdir()))

AUC 0.859 vs. 0.854; hits@243 -2 (-6 to +2) -> WORTH the out-of-fold runs (about 4 h)
saved: ['text_val_2025.csv', 'training_log_val_model.csv']


## Reading the results

**Training** took 26 minutes for one epoch: 12,928 rows, 604 positives, 4.0M trainable parameters. The held-out AUC (0.872, on 10% of the training rows) is close to the 2025 AUC, so the model isn't memorizing.

**LoRA-149M has the best AUC but the weakest very top:**
- **AUC 0.859,** the highest of the three text scores (fine-tuned small 0.854, frozen small 0.800). The margin over the fine-tuned small model, +0.005, is small and well within noise.
- **At the top 243 it finds 3,** against 5 and 4. The paired difference with the fine-tuned small model is −2, with a 90% range of −6 to +2: no real difference at this sample size.
- **Its strength is the broad ranking:** **38 of 52** exploited CVEs in its top 5,000, against 35 and 26. That's the property that matters when the text score is an *input* to LightGBM, which uses it to split groups of CVEs, rather than a ranking on its own.
- **It's less fooled by old bugs re-filed under new IDs:** 21 old-ID rows and 23 VulnCheck rows in its top 243, against 45 and 95 for the fine-tuned small model. That was the small model's main failure (`trackb_2c`).

**Decision:** by the rule fixed at the top, it qualifies (AUC above 0.854, and the hits@243 range reaches +2), so the out-of-fold runs go ahead (`trackb_8_lora_handoff_files.ipynb`). The AUC margin is thin, so the real test is whether it improves the **final model** (`trackb_9_lora_in_final_model.ipynb`). A better text score on its own is not enough.

*Timing note:* scoring 35,281 rows took 75 minutes at batch size 32. A later benchmark showed batch 8 is about 1.8× faster on the M3 with identical scores (max difference 5e-7), so `ml/text_model.py` now scores in batches of 8.